# 02 – Data Cleaning & Feature Engineering

Loads raw CSVs, removes duplicates/nulls, and engineers:
- Daily & log returns
- SMA / EMA / MACD
- Rolling volatility
- Lag features

**Output:** processed CSVs saved to `data/processed/<TICKER>_processed.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collection import StockDataCollector
from src.preprocessing import StockPreprocessor, load_processed

print('✅ Imports OK')

## 2.1 Load raw data

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']
RAW_DIR = '../data/raw'

raw_data = {}
for ticker in TICKERS:
    path = f'{RAW_DIR}/{ticker}.csv'
    if os.path.exists(path):
        raw_data[ticker] = pd.read_csv(path, index_col='Date', parse_dates=True)
        print(f'  Loaded {ticker}: {raw_data[ticker].shape}')
    else:
        print(f'  ⚠️  {ticker} not found – run notebook 01 first')

## 2.2 Run preprocessing pipeline

In [ ]:
processed = {}
for ticker, df in raw_data.items():
    proc = StockPreprocessor(df, ticker=ticker)
    processed[ticker] = proc.run(save=True)
    
print(f'\n✅ Processed {len(processed)} ticker(s)')

## 2.3 Inspect engineered features

In [ ]:
aapl = processed['AAPL']
print('Columns:', list(aapl.columns))
aapl.tail(3)

## 2.4 Missing values after cleaning

In [ ]:
null_counts = aapl.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else '✅ No missing values')

## 2.5 Moving averages overlay

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(aapl.index, aapl['close'],  label='Close',   color='#ecf0f1', linewidth=1.2, alpha=0.8)
ax.plot(aapl.index, aapl['sma_20'], label='SMA 20',  color='#f1c40f', linewidth=1.5)
ax.plot(aapl.index, aapl['sma_50'], label='SMA 50',  color='#e67e22', linewidth=1.5)
ax.plot(aapl.index, aapl['sma_200'],label='SMA 200', color='#9b59b6', linewidth=1.5)

ax.set_title('AAPL – Close Price & Moving Averages', fontsize=14, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/charts/AAPL_moving_averages.png', dpi=150)
plt.show()

## 2.6 Return distribution

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, (ticker, df) in enumerate(processed.items()):
    if i >= len(axes): break
    returns = df['daily_return'].dropna()
    axes[i].hist(returns, bins=60, color='#3498db', edgecolor='none', alpha=0.75)
    axes[i].set_title(ticker, fontsize=11)
    axes[i].set_xlabel('Daily Return')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Daily Return Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/charts/return_distributions.png', dpi=150)
plt.show()

---
**Next →** `03_eda_analysis.ipynb`